# 수도권제1순환고속도로 속도 예측 — Google Colaboratory
>⚠️ Google Colaboratory 런타임 환경 기준으로 작성되어 있습니다.

## 런타임 설정

Colab(T4 GPU)과 CPU 전용 환경 모두에서 동일한 노트북을 사용할 수 있도록 환경 구성을 자동화했습니다.

In [ ]:
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

def run_command(command, *, env=None):
    if isinstance(command, str):
        printable = command
    else:
        printable = ' '.join(str(c) for c in command)
    print(f'$ {printable}')
    subprocess.run(command, check=True, shell=isinstance(command, str), env=env)

def require(package: str, *, import_name: str | None = None, minimum: str | None = None, spec: str | None = None) -> None:
    name = import_name or package
    try:
        module = importlib.import_module(name)
        if minimum is not None:
            from packaging import version

            current = getattr(module, '__version__', None)
            if current is None or version.parse(current) < version.parse(minimum):
                raise ImportError
    except Exception:
        pkg_spec = spec or (f'{package}>={minimum}' if minimum else package)
        pip_packages.append(pkg_spec)

try:
    IS_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IS_COLAB = False
PROJECT_ROOT = Path.cwd().resolve()

In [ ]:
if IS_COLAB:
    apt_packages = ['htop', 'fonts-nanum']
    run_command('apt-get update')
    run_command('apt-get install -y ' + ' '.join(apt_packages))
else:
    print('ℹ️ Skipping apt-get because we are not running inside Google Colab.')

pip_packages: list[str] = []

require('matplotlib', minimum='3.8')
require('polars', minimum='1.6.0')
require('xlsxwriter')
require('fastexcel')
require('openpyxl')
require('htop')
require('nbformat', minimum='5.10')
require('nbclient', minimum='0.10')
require('torch', minimum='2.6')
require('tensordict')
require('torchrl')
require('triton')

if pip_packages:
    run_command([sys.executable, '-m', 'pip', 'install', '--upgrade', '--prefer-binary', *pip_packages])
else:
    print('ℹ️ Required Python packages already installed.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import matplotlib.font_manager as fm
    import matplotlib.pyplot as plt
except ImportError:
    print('ℹ️ Matplotlib not available after installation; skipping font configuration.')
else:
    mpl_cache = Path.home() / '.cache' / 'matplotlib'
    if mpl_cache.exists():
        shutil.rmtree(mpl_cache)

    if any('NanumGothic' in f.name for f in fm.fontManager.ttflist):
        plt.rc('font', family='NanumGothic')
    else:
        print('ℹ️ NanumGothic font not found; using default Matplotlib font.')
    plt.rcParams['axes.unicode_minus'] = False

print('✅ Setup complete.')

In [ ]:
run_command([sys.executable, '-m', 'pip', 'install', '--upgrade', '--prefer-binary', 'torchscale', 'transformer-engine-cu12', 'transformer-engine-torch'])
print('✅ Optional setup complete.')

In [ ]:
import torch

device_note = 'CUDA' if torch.cuda.is_available() else 'CPU/MPS'
print(f'✅ PyTorch {torch.__version__} imported successfully ({device_note} runtime).')

import os, multiprocessing as mp

# OpenMP/oneMKL: 스레드/affinity 경고/과다 스레드 방지
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("KMP_AFFINITY", "disabled")  # 경고 #226 억제
os.environ.setdefault("KMP_WARNINGS", "0")

# Polars 스레드풀 (임포트 전에만 유효)
os.environ.setdefault("POLARS_MAX_THREADS", "4")

## 모델 및 PyTorch 로드

작업 디렉터리와 RAW 데이터 위치를 환경 변수로 조정할 수 있습니다.

In [ ]:
import os

if IS_COLAB:
    from google.colab import drive

    drive.mount('/content/drive', force_remount=False)
    print('🔁 Google Drive를 마운트했습니다.')

os.chdir(os.path.join(os.getcwd(), "drive/My Drive/Colab Notebooks/수도권제1순환고속도로 통행 속도 추세 분석하기 (vNext)/"))

import torch
import stnet
from stnet.backend.environment import get_device

device = get_device()
print(f'✅ Torch initialized with {device.type.upper()} (PyTorch {torch.__version__}).')

## RAW 데이터 로드

모든 시트를 Polars LazyFrame으로 적재하여 후속 전처리에 활용합니다.

In [ ]:
import os
import polars as pl
import openpyxl

xlsx_path = os.path.join(os.getcwd(), "수도권제1순환고속도로.xlsx")
workbook = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
sheet_names = workbook.sheetnames

def _read_lazy(sheet_name: str) -> pl.LazyFrame:
    return pl.read_excel(xlsx_path, sheet_name=sheet_name).lazy()

raw_data = {sheet_name: _read_lazy(sheet_name) for sheet_name in sheet_names}
workbook.close()
print(f'✅ {len(sheet_names)}개 시트를 로드했습니다: {xlsx_path}')

## 피처, 라벨 인코딩

In [ ]:
import re
from typing import List, Sequence, Tuple, Union

import numpy as np

def _is_hour_col(name: str) -> bool:
    s = str(name)
    return len(s) == 3 and s.endswith('시') and s[:2].isdigit()

def schema_names(df: Union[pl.DataFrame, pl.LazyFrame]) -> List[str]:
    if isinstance(df, pl.DataFrame):
        return list(df.columns)
    if isinstance(df, pl.LazyFrame):
        return list(df.collect_schema().names())
    raise TypeError('df must be a Polars DataFrame or LazyFrame.')

def parse(name: str) -> Tuple[int, str]:
    m = re.search(r'^\d+', name)
    month = int(m.group()) if m else None
    daytype = '평일' if '평일' in name else '주말·공휴일'
    return month, daytype

def cleanse(
    df: Union[pl.DataFrame, pl.LazyFrame],
    month: int,
    daytype: str,
) -> pl.DataFrame:
    hour_cols = [c for c in HOURS if c in schema_names(df)]
    lazy_df = df.lazy() if isinstance(df, pl.DataFrame) else df

    return (
        lazy_df
        .with_columns(
            pl.col('구간').cast(pl.String),
            pl.col('방향').cast(pl.String),
        )
        .filter(pl.col('구간').is_not_null() & pl.col('구간').str.contains('-', literal=True))
        .with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in hour_cols])
        .with_columns(
            pl.col('구간').str.replace_all(r'\s+', '').alias('구간'),
            pl.col('방향').str.strip_chars().alias('방향'),
            pl.col('구간').str.split('-').list.sort().list.join('-').alias('구간ID'),
        )
        .unpivot(
            index=['구간', '구간ID', '방향'],
            on=hour_cols,
            variable_name='시간문자',
            value_name='속도',
        )
        .with_columns(
            pl.lit(month).alias('월'),
            pl.lit(DAY2ID[daytype]).alias('요일타입_id'),
            pl.when(pl.col('방향') == '상행').then(0)
              .when(pl.col('방향') == '하행').then(1)
              .otherwise(None)
              .cast(pl.Int8, strict=False)
              .alias('방향_id'),
            pl.col('시간문자').str.replace_all('시', '').cast(pl.Int64, strict=False).alias('시간'),
            pl.col('속도').cast(pl.Float64, strict=False),
        )
        .select(['월', '요일타입_id', '방향_id', '구간ID', '시간', '속도'])
        .filter(pl.col('방향_id').is_not_null())
        .group_by(['월', '요일타입_id', '방향_id', '구간ID', '시간'])
        .agg(pl.col('속도').mean().alias('속도'))
        .collect()
    )

def nanmean_grid(grid: np.ndarray) -> np.ndarray:
    with np.errstate(invalid='ignore'):
        col_mean = np.nanmean(grid, axis=0, keepdims=True)
    col_mean = np.where(np.isnan(col_mean), 0.0, col_mean)
    return np.where(np.isnan(grid), col_mean, grid)

In [ ]:
# === 셀 단위 회귀 데이터셋 (Polars Lazy) — 내부 파이프라인이 타깃 스케일을 일관 처리 ===
HOURS = [f'{h:02d}시' for h in range(24)]
DAY2ID = {'평일': 0, '주말·공휴일': 1}
DIR2ID = {'상행': 0, '하행': 1}
DIR_ID2NAME = {v: k for k, v in DIR2ID.items()}
long_parts: List[pl.DataFrame] = []

# 시트별 정제/롱화
for name in sheet_names:
    month, daytype = parse(name)
    if month is None:
        raise ValueError(f'시트명에서 월 정보를 찾을 수 없습니다: {name}')
    long_parts.append(cleanse(raw_data[name], month, daytype))

# 전체 연결 후 Lazy
df = pl.concat(long_parts, how='vertical', rechunk=True)
lf = df.lazy()

# 상행 기준 세그먼트 축(38) 확정, 하행 역순 매핑 (빈칸 방지)
up_list = (
    lf.filter(pl.col("방향_id") == 0)
      .select("구간ID").unique()
      .sort("구간ID")
      .collect()["구간ID"].to_list()
)
S, T = len(up_list), 24
up_map = pl.DataFrame({ "구간ID": up_list,        "seg_idx_up": list(range(S)) })
dn_map = pl.DataFrame({ "구간ID": up_list[::-1],  "seg_idx_dn": list(range(S)) })

lf_idxed = (
    lf.join(up_map.lazy(), on="구간ID", how="left", suffix="_up")
      .join(dn_map.lazy(), on="구간ID", how="left", suffix="_dn")
      .with_columns(pl.coalesce([pl.col("seg_idx_up"), pl.col("seg_idx_dn")]).alias("seg_idx"))
      .select(["월","요일타입_id","방향_id","구간ID","시간","seg_idx","속도"])
)

# --- 셀 단위 회귀 샘플 ---
# 주기 인코딩: 월/시간, 방향·요일은 numeric, seg_idx는 [-1,1] 스케일
PI = np.pi
features_lf = (
    lf_idxed
    .with_columns([
        pl.sin(2*PI*(pl.col("월").cast(pl.Float64)/12.0)).alias("month_sin"),
        pl.cos(2*PI*(pl.col("월").cast(pl.Float64)/12.0)).alias("month_cos"),
        pl.sin(2*PI*(pl.col("시간").cast(pl.Float64)/24.0)).alias("hour_sin"),
        pl.cos(2*PI*(pl.col("시간").cast(pl.Float64)/24.0)).alias("hour_cos"),
        pl.col("요일타입_id").cast(pl.Float64).alias("daytype"),
        pl.col("방향_id").cast(pl.Float64).alias("direction"),
        ((pl.col("seg_idx").cast(pl.Float64)/(pl.lit(float(S-1))))*2 - 1).alias("seg_norm"),
    ])
    .select(["month_sin","month_cos","hour_sin","hour_cos","daytype","direction","seg_norm"])
)
labels_lf = lf_idxed.select([pl.col("속도").cast(pl.Float64).alias("y")])
keys_lf = lf_idxed.select(["월","요일타입_id","방향_id","seg_idx","시간","구간ID"])

features_df = features_lf.collect()
labels_df = labels_lf.collect()
keys_df = keys_lf.collect()

features_tensor = torch.from_numpy(features_df.to_numpy()).to(torch.float32)   # (N,7)
label_tensor   = torch.from_numpy(labels_df.to_numpy()).to(torch.float32).unsqueeze(-1)  # (N,1)

# 키(월,요일,방향,seg_idx,시간)와 구간 텍스트
keys_sorted = [ (int(r[0]), int(r[1]), int(r[2]), int(r[3]), int(r[4])) for r in keys_df.select(["월","요일타입_id","방향_id","seg_idx","시간"]).iter_rows() ]
seg_texts  = keys_df["구간ID"].to_list()

label_shape = (1,)
print(f'✅ 셀 단위 회귀 샘플 수={len(keys_sorted)}, in_dim={features_tensor.shape[1]}, label_shape={label_shape}')

## 모델 생성

`stnet.api.config.ModelConfig`와 `stnet.api.config.PatchConfig`는 `modeling_type` 인자를 통해 입력 격자의 도메인을 설정합니다.
런타임 모듈에서도 동일한 변수들을 재노출하므로 기존 코드를 그대로 사용할 수 있습니다.
다음 별칭은 모두 대소문자와 무관하게 지원됩니다.

- 공간 전용: `ss`, `spatial`
- 시간 전용: `tt`, `temporal`
- 시공간 결합: `st`, `ts`, `spatiotemporal`


In [ ]:
from stnet.api.config import PatchConfig, ModelConfig

patch = PatchConfig(
    is_cube=True,
    grid_size_3d=(T, S, 1),
    patch_size_3d=(1, 1, 1),
    use_padding=True,
)

if device.type == 'cuda':
    config = ModelConfig(
        device=device,
        microbatch=64,
        dropout=0.05,
        normalization_method='rmsnorm',
        depth=1152,
        heads=6,
        spatial_depth=4,
        temporal_depth=4,
        mlp_ratio=3.0,
        drop_path=0.05,
        spatial_latents=80,
        temporal_latents=60,
        modeling_type='spatiotemporal',
        patch=patch,
        compile_mode='max-autotune',
    )
else:
    config = ModelConfig(
        device=device,
        microbatch=32,
        dropout=0.10,
        normalization_method='layernorm',
        depth=512,
        heads=2,
        spatial_depth=2,
        temporal_depth=2,
        mlp_ratio=2.0,
        drop_path=0.0,
        spatial_latents=32,
        temporal_latents=32,
        modeling_type='spatiotemporal',
        patch=patch,
        compile_mode='max-autotune-no-cudagraphs',
    )

print(config)

In [ ]:
from stnet.api.io import new_model

in_dim = 7
out_shape = (1,)
model = new_model(config=config, in_dim=in_dim, out_shape=out_shape)
print(model)

##모델 학습

In [ ]:
num_samples = features_tensor.shape[0]
base_params = {
    'epochs': 100 if device.type == 'cuda' else 50,
    'batch_size': 8,
    'base_lr': 3e-3,
    'weight_decay': 1e-4,
}
print(base_params)

In [ ]:
import os
os.environ["WORLD_SIZE"] = "1"  # 분산 비활성 힌트
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"

from stnet.api.run import train

train_dataset = {"X": features_tensor, "Y": label_tensor}
model = train(
    model,
    train_dataset,
    epochs=int(base_params["epochs"]),
    batch_size=int(base_params["batch_size"]),
    base_lr=float(base_params["base_lr"]),
    weight_decay=float(base_params["weight_decay"]),
    val_frac=0.1,
)

In [ ]:
print("keys:", len(keys_sorted), "label_shape:", label_shape)

##추론 데이터를 저장할 템플릿 생성

In [ ]:
import copy

template = copy.deepcopy(raw_data)
time_columns = [col for col in schema_names(next(iter(template.values()))) if _is_hour_col(col)]
template = {
    sheet_name: lf.with_columns([pl.lit(None).alias(col) for col in time_columns])
    for sheet_name, lf in template.items()
}

## 모델 추론 및 평가


In [ ]:
from stnet.api.run import predict

infer_samples = {key: None for key in keys_sorted}
raw_predictions = predict(
    model,
    infer_samples,
    batch_size=int(base_params["batch_size"]),
)

pred_tensor = torch.stack([raw_predictions[key] for key in keys_sorted]).detach().cpu()  # (N,1)
target_tensor = label_tensor.detach().cpu()

mae = torch.mean(torch.abs(pred_tensor - target_tensor)).item()
rmse = torch.sqrt(torch.mean((pred_tensor - target_tensor) ** 2)).item()
print(f'✅ Evaluation — MAE: {mae:.3f}, RMSE: {rmse:.3f}')

# 롱 포맷 재구성 (노출용)
pred_np = pred_tensor.squeeze(-1).numpy()
out_long = pl.DataFrame({
    "월": [k[0] for k in keys_sorted],
    "요일타입_id": [k[1] for k in keys_sorted],
    "방향_id": [k[2] for k in keys_sorted],
    "seg_idx": [k[3] for k in keys_sorted],
    "시간": [k[4] for k in keys_sorted],
    "구간ID": seg_texts,
    "속도": pred_np,
}).with_columns([
    pl.when(pl.col("요일타입_id")==0).then("평일").otherwise("주말·공휴일").alias("요일타입"),
    pl.when(pl.col("방향_id")==0).then("상행").otherwise("하행").alias("방향"),
])

def long_to_wide(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.select(["구간ID","방향","시간","속도"])
          .pivot(index=["구간ID","방향"], columns="시간", values="속도")
          .rename({i: f"{i:02d}시" for i in range(24)})
          .sort(["방향","구간ID"])
    )

pred_sheets = {}
for sheet_name in sheet_names:
    m, daytype = parse(sheet_name)
    d_id = DAY2ID[daytype]
    sub = out_long.filter( (pl.col("월")==m) & (pl.col("요일타입_id")==d_id) )
    pred_sheets[sheet_name] = long_to_wide(sub)

import xlsxwriter, os
pred_xlsx = os.path.join(os.getcwd(), "예측 데이터_cell_reg_integrated.xlsx")
workbook = xlsxwriter.Workbook(str(pred_xlsx))
for sheet_name, frame in pred_sheets.items():
    frame.write_excel(workbook=workbook, worksheet=sheet_name, autofit=True)
workbook.close()
print(f'✅ 예측 결과 저장: {pred_xlsx}')

## 추론 데이터 저장

In [ ]:
from typing import Dict, Sequence, Tuple

def get_schema(raw_data: Dict[str, pl.LazyFrame]) -> Tuple[Sequence[str], Sequence[str]]:
    first_sheet = next(iter(raw_data.values()))
    cols = schema_names(first_sheet)
    idx_first_hour = min([i for i, c in enumerate(cols) if _is_hour_col(c)], default=len(cols))
    schema_left = list(cols[:idx_first_hour]) if idx_first_hour > 0 else ['구간', '방향']
    schema_right = HOURS
    return (schema_left, schema_right)

def categorize_poi(seg_id: str, dir_id: int, delim: str = '-') -> str:
    a, b = seg_id.split(delim, 1)
    return f'{a}{delim}{b}' if int(dir_id) == 0 else f'{b}{delim}{a}'

def to_long_polars(key: Tuple[int, int, int], grid: torch.Tensor, seg_ids: Sequence[str]) -> pl.DataFrame:
    m, day_id, dir_id = map(int, key)
    S_dim, T_dim, C_dim = grid.shape
    assert C_dim == 1
    g = grid.detach().to('cpu').float().numpy()
    recs = [
        (m, day_id, dir_id, seg_ids[s], t, float(g[s, t, 0]))
        for s in range(S_dim)
        for t in range(T_dim)
    ]
    return pl.DataFrame(recs, schema=['월', '요일타입_id', '방향_id', '구간ID', '시간', '속도'], orient='row')

def to_wide_polars(long_df: pl.DataFrame, index_cols: Sequence[str]) -> pl.DataFrame:
    long_df = long_df.with_columns(pl.col("시간").cast(pl.Int64, strict=False))

    long_df = long_df.with_columns(pl.col("시간").cast(pl.Utf8))
    w = long_df.pivot(values="속도", index=index_cols, on="시간", aggregate_function="mean")
    current_hours = {c for c in w.columns if c not in index_cols}
    missing = [str(h) for h in range(24) if str(h) not in current_hours]
    if missing:
        w = w.with_columns([pl.lit(None).alias(h) for h in missing])
    w = w.select(list(index_cols) + [str(h) for h in range(24)])
    w = w.rename({str(h): f"{h:02d}시" for h in range(24)})
    return w.with_columns(
        [pl.col('방향_id').cast(pl.Int64, strict=False),
         pl.col('구간ID').cast(pl.String)]
        + [pl.col(c).cast(pl.Float64, strict=False).alias(c) for c in HOURS]
    )

def to_polars(
    template_lf: pl.LazyFrame,
    schema_left: Sequence[str],
    schema_right: Sequence[str],
    pred_long_for_sheet: pl.DataFrame,
) -> pl.DataFrame:
    temp = template_lf.collect()
    if '구간' not in temp.columns or '방향' not in temp.columns:
        raise ValueError("템플릿 시트에 '구간' 또는 '방향' 컬럼이 없습니다.")

    def _map_dir(s: str) -> int:
        if s not in DIR2ID:
            raise ValueError(f"[방향 라벨] 알 수 없는 값: {s!r}")
        return DIR2ID[s]

    def _norm_seg(s: str) -> str:
        s = re.sub(r"\s+", "", str(s))
        parts = s.split('-')
        parts = sorted([p for p in parts if p != ""])
        return "-".join(parts)

    temp = temp.with_columns([
        pl.col('구간').map_elements(_norm_seg, return_dtype=pl.String).alias('구간ID'),
        pl.col('방향').map_elements(_map_dir, return_dtype=pl.Int64).alias('방향_id'),
    ])

    if pred_long_for_sheet.height == 0:
        raise AssertionError("[예측 롱] 이 시트(월/요일타입) 조합의 예측이 비어 있습니다. DAY2ID/월 필터를 확인하세요.")

    wide_pred = to_wide_polars(pred_long_for_sheet, index_cols=['구간ID', '방향_id'])
    wide_pred = wide_pred.with_columns([
        pl.col('구간ID').cast(pl.String),
        pl.col('방향_id').cast(pl.Int64, strict=False),
    ])
    temp = temp.with_columns([
        pl.col('구간ID').cast(pl.String),
        pl.col('방향_id').cast(pl.Int64, strict=False),
    ])
    keys = ['구간ID', '방향_id']
    wide_pred = wide_pred.select(
        [pl.col(k) for k in keys] +
        [pl.col(c).alias(f"pred_{c}") for c in wide_pred.columns if c not in keys]
    )
    dup_r = wide_pred.group_by(keys).len().filter(pl.col('len') > 1).height
    assert dup_r == 0, f"[예측 키중복] wide_pred에 중복 키 존재 (rows>1) : {dup_r}"
    dup_l = temp.group_by(keys).len().filter(pl.col('len') > 1).height
    assert dup_l == 0, f"[템플릿 키중복] temp에 중복 키 존재 (rows>1) : {dup_l}"
    joined = temp.join(wide_pred, on=keys, how='left')
    final = joined
    fill_exprs = []
    for h in schema_right:
        pred_col = f'pred_{h}'
        base = pl.col(h) if h in final.columns else pl.lit(None)
        src  = pl.col(pred_col) if pred_col in final.columns else pl.lit(None)
        fill_exprs.append(pl.when(src.is_not_null()).then(src).otherwise(base).alias(h))
    final = final.with_columns(fill_exprs)
    final = final.drop([c for c in final.columns if c.startswith('pred_')] + ['구간ID', '방향_id'])
    keep_left = [c for c in schema_left if c in final.columns]
    final = final.select(keep_left + [c for c in schema_right if c in final.columns])
    cols = [h for h in schema_right if h in final.columns]
    _nulls = final.select([pl.col(h).is_null().sum().alias(h) for h in cols])
    _total = int(_nulls.select(pl.sum_horizontal(*[pl.col(h) for h in _nulls.columns])).item())
    _nans = final.select([pl.col(h).is_nan().sum().alias(h) for h in HOURS if h in final.columns])
    _infs = final.select([
      ((pl.col(h) == float('inf')) | (pl.col(h) == float('-inf'))).sum().alias(h)
    for h in HOURS if h in final.columns
    ])
    def _total(df: pl.DataFrame) -> int:
       if df.height == 0:
           return 0
       vals = df.row(0)
       s = 0
       for v in vals:
            if isinstance(v, (int, float)):
                s += 0 if v is None else int(v)
       return s
    tn, ti = _total(_nans), _total(_infs)
    if tn or ti:
        print("[NaN/Inf 발생 요약] NaN:", tn, " Inf:", ti)
        bad = final.select(['구간', '방향'] + [h for h in HOURS if h in final.columns]).filter(
            pl.any_horizontal([pl.col(h).is_nan() | (pl.col(h)==float('inf')) | (pl.col(h)==float('-inf'))
                               for h in HOURS if h in final.columns])
        )
        print(bad.head(10))
    if _total(_nulls) > 0:
        missing = temp.select(keys).join(wide_pred.select(keys), on=keys, how='anti')
        print("[누락 키 상위 20개]")
        print(missing.head(20))
        total_nulls = _total(_nulls)
        raise AssertionError(f"[병합 누락] 시간열 NULL 총 {total_nulls}개. 열별 합계:\n{_nulls}")
    final = final.with_columns([
        pl.col(h).cast(pl.Float64, strict=False).alias(h)
        for h in cols
    ])
    return final

In [ ]:
pred_longs = []
keys_sorted = sorted(pred_dict.keys())
for key in keys_sorted:
    grid = pred_dict[key]
    pred_longs.append(to_long_polars(key, grid, segments))
pred_long_df = pl.concat(pred_longs, how='vertical', rechunk=True)

schema_left, schema_right = get_schema(template)

pred_sheets: Dict[str, pl.DataFrame] = {}
for sheet_name, lf in template.items():
    month = int(re.search(r'^\d+', sheet_name).group())
    daytype = '평일' if '평일' in sheet_name else '주말·공휴일'
    d_id = DAY2ID[daytype]
    long_for_sheet = pred_long_df.filter((pl.col('월') == month) & (pl.col('요일타입_id') == d_id))
    nan_long = int(long_for_sheet.select(pl.col("속도").is_nan().sum()).item())
    if nan_long > 0:
        print(f"[예측 롱] NaN: {nan_long}")
        print(long_for_sheet.filter(pl.col("속도").is_nan())
          .select(["구간ID","방향_id","시간"]).head(10))
    final_df = to_polars(lf, schema_left=schema_left, schema_right=schema_right, pred_long_for_sheet=long_for_sheet)
    pred_sheets[sheet_name] = final_df
    nan_long = int(long_for_sheet.select(pl.col("속도").is_nan().sum()).item())
    if nan_long > 0:
        print(f"[예측 롱] NaN 개수: {nan_long}")
        print(long_for_sheet.filter(pl.col("속도").is_nan()).head(10))

In [ ]:
import xlsxwriter
import os

pred_xlsx = os.path.join(os.getcwd(), "예측 데이터.xlsx")
workbook = xlsxwriter.Workbook(str(pred_xlsx))

for sheet_name, frame in pred_sheets.items():
    frame.write_excel(workbook=workbook, worksheet=sheet_name, autofit=True)

workbook.close()
print(f'✅ 예측 결과를 저장했습니다: {pred_xlsx}')